# R3-03 · RAG — Profundización (opcional) 🔬

**Formación Pública — Rama R3 · IA Aplicada · Notebook de profundización**

Opcional. En la lección la recuperación acertó la ficha correcta al primer intento en
**la mitad** de las preguntas (hit@1 = 0,50). Aquí la vas a mejorar como se hace en serio:
**un cambio a la vez, midiendo cada uno** con el mismo set de preguntas.

Palancas que vas a probar: **peso del título**, **n-gramas** (de palabras y de caracteres) y
**tamaño del trozo**. Todo corre sin red ni LLM: es pura recuperación.

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import os, urllib.request
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

CSV = "fichas.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/fichas.csv", CSV)
fichas = pd.read_csv(CSV)

# Palabras vacías (sin tilde, porque el vectorizador quita las tildes antes de comparar)
STOPWORDS = """a al algo como con cual cuales cuando de del desde donde e el en entre es esta este esto
estos fue ha hay la las le les lo los mas me mi mis muy ni no o para pero por que quien se sea
segun si sin sobre son su sus te tiene tu tus un una uno unos y ya yo""".split()

def trocear(texto, max_palabras=120):
    trozos, actual, n = [], [], 0
    for linea in texto.split("\n"):
        if not linea.strip():                    # las líneas en blanco no aportan
            continue
        palabras = len(linea.split())
        if actual and (linea.startswith("## ") or n + palabras > max_palabras):
            trozos.append("\n".join(actual))
            actual, n = [], 0
        actual.append(linea)
        n += palabras
    if actual:
        trozos.append("\n".join(actual))
    return trozos

def construir_indice(max_palabras=120, peso_titulo=1, **opciones):
    """Trocea (max_palabras=None = ficha completa), antepone el título `peso_titulo` veces
    y vectoriza con TF-IDF.
    `opciones` se pasan tal cual a TfidfVectorizer (ej. ngram_range, analyzer)."""
    filas = [{"id": f.id, "titulo": f.titulo, "url": f.url, "texto": t}
             for f in fichas.itertuples()
             for t in ([f.texto] if max_palabras is None else trocear(f.texto, max_palabras))]
    trozos = pd.DataFrame(filas)
    documentos = (trozos["titulo"] + "\n") * peso_titulo + trozos["texto"]
    opciones = {"strip_accents": "unicode", **opciones}
    if opciones.get("analyzer", "word") == "word":
        opciones.setdefault("stop_words", STOPWORDS)
    vect = TfidfVectorizer(**opciones)
    return {"trozos": trozos, "vect": vect, "X": vect.fit_transform(documentos)}

def recuperar(indice, pregunta, k=3):
    puntajes = (indice["X"] @ indice["vect"].transform([pregunta]).T).toarray().ravel()
    mejores = np.argsort(-puntajes, kind="stable")[:k]
    res = indice["trozos"].iloc[mejores].copy()
    res["puntaje"] = puntajes[mejores]
    return res

def hit_at_k(indice, k, preguntas=None):
    preguntas = preguntas or PREGUNTAS_EVAL
    return sum(e in set(recuperar(indice, p, k)["id"]) for p, e in preguntas) / len(preguntas)

# Set de evaluación: preguntas redactadas como las haría una persona en la ventanilla,
# con la ficha que DEBERÍA aparecer (id de fichas.csv). Revisadas a mano una por una.
PREGUNTAS_EVAL = [
    ("¿Cómo saco el certificado de antecedentes?", 3442),
    ("¿Cómo me afilio a Fonasa?", 9715),
    ("¿Cómo obtengo un certificado de alumno regular?", 33265),
    ("¿Qué necesito para celebrar el acuerdo de unión civil?", 37532),
    ("¿Cómo saco la licencia para pescar como deporte?", 4636),
    ("¿Cuándo pagan el aguinaldo de Navidad a los pensionados?", 30171),
    ("¿Quiénes reciben el bono de invierno?", 39484),
    ("¿Cómo renuevo mi carnet si vivo fuera de Chile?", 48164),
    ("Me rechazaron la licencia médica porque la presenté tarde, ¿cómo apelo?", 9620),
    ("¿Cómo aviso que voy a hacer una quema en mi campo?", 742),
    ("¿Cómo busco trabajo en la bolsa de empleo del Estado?", 5072),
    ("¿Cómo cobro el fondo de cesantía solidario?", 36646),
    ("¿Cómo obtengo el certificado de discapacidad?", 83219),
    ("¿Qué es el bono marzo?", 38913),
    ("¿Dónde compro la entrada a un parque nacional?", 107334),
    ("Necesito un abogado gratis para divorciarme", 316),
    ("¿Cómo saco la ClaveÚnica estando en el extranjero?", 44751),
    ("Me despidieron injustamente, ¿dónde reclamo?", 120584),
    ("¿Cómo pido la pensión de alimentos para mis hijos?", 313),
    ("¿Cómo obtengo la tarjeta de alimentación para la universidad?", 2125),
]
print("Preguntas de evaluación:", len(PREGUNTAS_EVAL))

## 1. La línea base (y cuánto vale el título)

Primero, el punto de partida: el índice de la lección (`construir_indice()` con sus valores por defecto: trozos de 120 palabras, título una vez, palabras sueltas). Después, la misma receta **sin** título (`peso_titulo=0`). Nunca cambies algo sin saber desde dónde partes.

### ✍️ Ejercicio 1 — Mide la línea base y la versión sin título

In [ ]:
base = construir_indice()
sin_titulo = construir_indice(peso_titulo=0)
resultados = {
    "base":       (hit_at_k(base, 1), hit_at_k(base, 3)),
    "sin_titulo": (hit_at_k(sin_titulo, 1), hit_at_k(sin_titulo, 3)),
}
for nombre, (h1, h3) in resultados.items():
    print(f"{nombre:12s} hit@1={h1:.2f}  hit@3={h3:.2f}")

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert resultados["base"] == (hit_at_k(base, 1), hit_at_k(base, 3))
    assert resultados["sin_titulo"] == (hit_at_k(sin_titulo, 1), hit_at_k(sin_titulo, 3))
    assert resultados["sin_titulo"][0] < resultados["base"][0], "sin título, hit@1 debería caer"
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'sin_titulo = construir_indice(peso_titulo=0); cada tupla es (hit_at_k(indice, 1), hit_at_k(indice, 3)).')
    print("   Detalle:", e)

## 2. Títulos con más peso

Si el título ayuda tanto, ¿y si pesa más? Repetir el título `n` veces es un truco simple para que sus palabras cuenten más en TF-IDF. Prueba `peso_titulo` en 1, 2, 3 y 5 y quédate con el mejor (primero por hit@1; si empatan, por hit@3).

### ✍️ Ejercicio 2 — Encuentra el mejor `peso_titulo`

In [ ]:
por_peso = {}
for peso in [1, 2, 3, 5]:
    ind = construir_indice(peso_titulo=peso)
    por_peso[peso] = (hit_at_k(ind, 1), hit_at_k(ind, 3))
    print(f"peso {peso}: hit@1={por_peso[peso][0]:.2f}  hit@3={por_peso[peso][1]:.2f}")
mejor_peso = max(por_peso, key=lambda p: por_peso[p])
print("mejor peso:", mejor_peso)

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert set(por_peso) == {1, 2, 3, 5}
    assert por_peso[mejor_peso] == max(por_peso.values())
    assert por_peso[mejor_peso][0] > resultados["base"][0], "el mejor peso debería superar a la línea base en hit@1"
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'Dentro del for: ind = construir_indice(peso_titulo=peso); por_peso[peso] = (hit_at_k(ind, 1), hit_at_k(ind, 3)).')
    print("   Detalle:", e)

## 3. N-gramas: pares de palabras y trozos de palabra

Muchas fallas eran de **vocabulario**: «pescar» vs «pesca», «divorciarme» vs «divorcio». Dos variantes de TF-IDF atacan eso de distinta forma:

- `ngram_range=(1, 2)`: además de palabras sueltas, **pares** («licencia médica», «bono marzo»).
- `analyzer="char_wb", ngram_range=(3, 5)`: pedacitos de **3 a 5 letras** dentro de cada palabra. «pescar» y «pesca» comparten «pes», «esc», «pesc»… y se vuelven parecidas.

Mide ambas con el mejor peso de título.

### ✍️ Ejercicio 3 — Compara bigramas y n-gramas de caracteres

In [ ]:
bigramas = construir_indice(peso_titulo=mejor_peso, ngram_range=(1, 2))
caracteres = construir_indice(peso_titulo=mejor_peso, analyzer="char_wb", ngram_range=(3, 5))
por_ngrama = {
    "palabras (1,2)":   (hit_at_k(bigramas, 1), hit_at_k(bigramas, 3)),
    "caracteres (3,5)": (hit_at_k(caracteres, 1), hit_at_k(caracteres, 3)),
}
for nombre, (h1, h3) in por_ngrama.items():
    print(f"{nombre:17s} hit@1={h1:.2f}  hit@3={h3:.2f}")
print(recuperar(caracteres, "¿Cómo saco la licencia para pescar como deporte?", 1)["titulo"].iloc[0])

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert por_ngrama["palabras (1,2)"] == (hit_at_k(bigramas, 1), hit_at_k(bigramas, 3))
    assert por_ngrama["caracteres (3,5)"] == (hit_at_k(caracteres, 1), hit_at_k(caracteres, 3))
    assert por_ngrama["caracteres (3,5)"][1] > resultados["base"][1], "los n-gramas de caracteres deberían mejorar hit@3"
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'construir_indice(peso_titulo=mejor_peso, ngram_range=(1, 2)) y construir_indice(peso_titulo=mejor_peso, analyzer="char_wb", ngram_range=(3, 5)).')
    print("   Detalle:", e)

## 4. Otro tamaño de trozo

¿Trozos chicos (más precisos, pero con menos contexto) o grandes (más contexto, pero mezclan temas)? Con la mejor configuración hasta ahora (mejor peso + n-gramas de caracteres), prueba `max_palabras` = 60, 120, 250 y la **ficha completa** (`None` = no trocear).

### ✍️ Ejercicio 4 — Mide el efecto del tamaño de trozo

In [ ]:
filas = []
for mp in [60, 120, 250, None]:
    ind = construir_indice(max_palabras=mp, peso_titulo=mejor_peso, analyzer="char_wb", ngram_range=(3, 5))
    filas.append({"max_palabras": mp or "ficha completa", "trozos": len(ind["trozos"]),
                  "hit@1": hit_at_k(ind, 1), "hit@3": hit_at_k(ind, 3)})
tabla_trozos = pd.DataFrame(filas)
tabla_trozos

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert len(tabla_trozos) == 4 and list(tabla_trozos["max_palabras"])[:3] == [60, 120, 250]
    assert tabla_trozos["trozos"].is_monotonic_decreasing, "más palabras por trozo = menos trozos"
    completa = tabla_trozos.iloc[-1]
    assert completa["trozos"] == len(fichas), "sin trocear debería quedar un trozo por ficha"
    assert completa["hit@1"] < tabla_trozos["hit@1"].max(), "la ficha completa no debería ser la mejor opción"
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'ind = construir_indice(max_palabras=mp, peso_titulo=mejor_peso, analyzer="char_wb", ngram_range=(3, 5)).')
    print("   Detalle:", e)

## 5. Desafío: la tabla final y la trampa del sobreajuste

Junta todo en una tabla `comparacion` (una fila por configuración —incluidos los tamaños de trozo del ejercicio 4—, columnas `hit@1` y `hit@3`), ordénala de mejor a peor y elige la `ganadora`.

**La trampa:** elegiste la configuración mirando **las mismas 20 preguntas** con que la mides. Es como estudiar con la prueba en la mano. Antes de llevarla a producción, escribe preguntas **nuevas** (idealmente, de personas reales de tu servicio) y vuelve a medir.

### ✍️ Ejercicio 5 — Arma la comparación y elige la ganadora

In [ ]:
comparacion = pd.DataFrame(
    {"base (lección)": resultados["base"],
     "sin título": resultados["sin_titulo"],
     f"título x{mejor_peso}": por_peso[mejor_peso],
     **{f"título x{mejor_peso} + {k}": v for k, v in por_ngrama.items()},
     **{f"título x{mejor_peso} + caracteres + trozos de {f['max_palabras']}": (f["hit@1"], f["hit@3"])
        for f in tabla_trozos.to_dict("records") if f["max_palabras"] != 120}},   # 120 ya está arriba
    index=["hit@1", "hit@3"]).T.sort_values(["hit@1", "hit@3"], ascending=False)
ganadora = comparacion.index[0]
print(comparacion)
print("\nGanadora:", ganadora)

In [ ]:
# ── Celda de chequeo · Ejercicio 5 ──────────────────────────────────────────
try:
    assert {"hit@1", "hit@3"} <= set(comparacion.columns)
    assert len(comparacion) >= 4
    assert list(comparacion["hit@1"]) == sorted(comparacion["hit@1"], reverse=True)
    assert ganadora == comparacion.index[0]
    print("✅ Ejercicio 5: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "pd.DataFrame({nombre: (h1, h3), ...}, index=['hit@1','hit@3']).T.sort_values(['hit@1','hit@3'], ascending=False); ganadora = comparacion.index[0].")
    print("   Detalle:", e)

## Para llevar

- **Un cambio a la vez, medido**: así se mejora un RAG. Sin set de evaluación, cada «mejora» es una opinión.
- El **título** es el metadato más barato y el que más aporta: un trozo sin contexto no se encuentra.
- Los **n-gramas de caracteres** absorben variaciones de palabras (conjugaciones, plurales, errores
  de tipeo) sin diccionarios ni modelos pesados.
- Trozos **demasiado grandes** diluyen la señal; la ficha completa casi nunca es la mejor unidad.
- **20 preguntas son pocas** y las usaste para elegir: la cifra final está inflada. Agrega
  preguntas nuevas antes de confiar en ella.

---

*Fin de la profundización de R3-03.*